# 🛡️ FRONTIER-OS // C4ISR Master GPU Training Suite (Google Colab)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maykonlincolnusa/FRONTIER-OS-Frontier-Operating-System-/blob/main/notebooks/colab/00_FRONTIER_C4ISR_Master_Colab_Training_Suite.ipynb)

Este notebook executa o pipeline completo de treinamento em **GPU (CUDA Mixed Precision FP16)** para todos os **5 Subsistemas de Deep Learning** do **FRONTIER-OS**:
1. **`01-maritime-coastline` (USCG / US Navy):** `TrajectoryAnomalyLSTM` -> Exporta `maritime_lstm.onnx`
2. **`02-airspace-shield` (USAF / CBP AMO):** `FlightStateAutoencoder` -> Exporta `airspace_ae.onnx`
3. **`03-narcotics-intel` (DEA / CBP / HIDTA):** `SpatioTemporalGNN` -> Exporta `narcotics_stgnn.onnx`
4. **`04-criminal-interstate` (FBI / State Police):** `ArcFace-512 Biometrics & Qdrant Embeddings`
5. **`05-geospatial-recon` (US Army / NGA):** `YOLOv8-Military Satellite Detector` -> Exporta `frontier_sat_yolov8.onnx`


In [7]:
# 1. Verificar GPU (NVIDIA T4 / L4 / A100) e clonar o repositório oficial
!nvidia-smi
import os, sys

if not os.path.exists("FRONTIER-OS-Frontier-Operating-System-"):
    !git clone https://github.com/maykonlincolnusa/FRONTIER-OS-Frontier-Operating-System-.git

%cd FRONTIER-OS-Frontier-Operating-System-
!pip install -q torch torchvision torchaudio onnx onnxscript onnxruntime-gpu httpx tqdm geopandas shapely pyarrow matplotlib pydantic-settings
sys.path.insert(0, os.getcwd())
print("✅ Ambiente GPU configurado com sucesso!")

Sun Nov  8 13:58:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
# 2. Baixar Dados Reais das Agências do Governo dos EUA (Census TIGER, FBI Wanted, OpenSky, CBP, MarineCadastre)
!pip install -q pydantic-settings
!python3 infra/scripts/download_real_gov_datasets.py



PROJECT FRONTIER (FRONTIER-C4ISR) - OFFICIAL US GOV DATA INGESTION

--- [1/6] US CENSUS BUREAU TIGER/LINE BOUNDARIES & INTERSTATES ---
[14:25:14] [INFO] [GOV-DATA] Downloading cb_2022_us_state_20m.zip from https://www2.census.gov/geo/tiger/GENZ2022/shp/cb_2022_us_state_20m.zip ...
[14:25:14] [INFO] [GOV-DATA] HTTP Request: GET https://www2.census.gov/geo/tiger/GENZ2022/shp/cb_2022_us_state_20m.zip "HTTP/1.1 200 OK"
[14:25:14] [INFO] [GOV-DATA] Successfully downloaded cb_2022_us_state_20m.zip (0.19 MB)
[14:25:14] [INFO] [GOV-DATA] Downloading cb_2022_us_nation_20m.zip from https://www2.census.gov/geo/tiger/GENZ2022/shp/cb_2022_us_nation_20m.zip ...
[14:25:14] [INFO] [GOV-DATA] HTTP Request: GET https://www2.census.gov/geo/tiger/GENZ2022/shp/cb_2022_us_nation_20m.zip "HTTP/1.1 200 OK"
[14:25:14] [INFO] [GOV-DATA] Successfully downloaded cb_2022_us_nation_20m.zip (0.09 MB)
[14:25:14] [INFO] [GOV-DATA] Downloading tl_2023_us_primaryroads.zip from https://www2.census.gov/geo/tiger/TIGER202

In [10]:
# 3. Treinar Todos os 4 Modelos de Deep Learning em GPU e Exportar para ONNX (Dynamic Axes)
%pip install -q pydantic-settings pydantic

import time
t0 = time.time()

print("🚢 [1/4] Treinando Subsistema 01: Maritime AIS LSTM (USCG / US Navy)...")
!python3 ml_training/maritime_lstm/train_ais_lstm.py

print("\n✈️ [2/4] Treinando Subsistema 02: Airspace ADS-B Autoencoder (USAF / FAA)...")
!python3 ml_training/airspace_autoencoder/train_adsb_ae.py

print("\n⚠️ [3/4] Treinando Subsistema 03: Narcotics Spatio-Temporal GNN (DEA / CBP)...")
!python3 ml_training/narcotics_gnn/train_stgnn.py

print("\n🛰️ [4/4] Treinando Subsistema 05: Satellite Recon YOLOv8 (US Army / NGA)...")
!python3 ml_training/satellite_vision/train_yolo_military.py

print(f"\n✅ Todos os modelos treinados e exportados para ONNX em {time.time() - t0:.2f}s!")
!ls -lh ml-training/weights/

🚢 [1/4] Treinando Subsistema 01: Maritime AIS LSTM (USCG / US Navy)...
[MARITIME TRAIN] Generating preprocessed MarineCadastre AIS sequences...
[MARITIME TRAIN] Training TrajectoryAnomalyLSTM on: cuda
  Epoch 01/10 | Loss: 0.14106
  Epoch 05/10 | Loss: 0.02054
  Epoch 10/10 | Loss: 0.01057
/content/FRONTIER-OS-Frontier-Operating-System-/FRONTIER-OS-Frontier-Operating-System-/FRONTIER-OS-Frontier-Operating-System-/ml_training/maritime_lstm/train_ais_lstm.py:80: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W1108 14:47:21.021000 15587 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converti

In [ ]:
# 4. Validar Inferência ONNX Runtime e Empacotar Pesos (.zip) para Download ou Deploy OCI
import zipfile
from pathlib import Path
import onnxruntime as ort
import numpy as np

weights_dir = Path("ml-training/weights")
for onnx_file in weights_dir.glob("*.onnx"):
    sess = ort.InferenceSession(str(onnx_file), providers=["CPUExecutionProvider"])
    inputs = {inp.name: f"{inp.shape} ({inp.type})" for inp in sess.get_inputs()}
    print(f"✔️ ONNX Validado: {onnx_file.name} | Inputs: {inputs}")

zip_path = "frontier_c4isr_onnx_weights.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in weights_dir.iterdir():
        if f.is_file():
            zf.write(f, arcname=f.name)

print(f"\n📦 Arquivo pronto para download: {zip_path}")
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Execute no Google Colab para iniciar o download automático pelo navegador.")
